# Réseau piéton

In [ ]:
# Geneva Cycle & Pedestrian Network Analysis
# -------------------------------------------------
# This script downloads Open data for the Canton of Geneva. 
# Only run section 5. to dowload data and save the network into segments
# -------------------------------------------------

import osmnx as ox
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import LineString
from shapely.ops import nearest_points
from shapely.ops import unary_union
from shapely.geometry import Point
import networkx as nx
from shapely.ops import substring


## 1. Per sidewalk

In [ ]:
# -------------------------------------------------
# 1. Loading only ofr Geneva city (as sample)
# -------------------------------------------------
area_name = "Geneva, Switzerland"
print(f"Downloading administrative boundary for: {area_name}")

boundary = ox.geocode_to_gdf(area_name)

#Ensure CRS is set and in WGS84
if boundary.crs is None:
    boundary.set_crs("EPSG:4326", inplace=True)
else:
    boundary = boundary.to_crs("EPSG:4326")

#Combine into a single polygon (recommended)
polygon = boundary.geometry.union_all()

In [ ]:
'''
# -------------------------------------------------
# 1. Fetch the canton boundary and project to a Swiss metric CRS (LV95 / EPSG:2056)
# -------------------------------------------------

# Fetch the full Canton of Geneva by OSM ID
#Query the canton using name and admin level
query = {
    "boundary": "administrative",
    "admin_level": "4",  # Swiss cantons
    "name": "Genève"
}
boundary = ox.features_from_place("Switzerland", tags=query)

# Optional: check result
print(boundary[["name", "admin_level"]])

# Use the geometry of the first match
polygon = boundary.geometry.iloc[0]'''

In [ ]:
# -------------------------------------------------
# 2. Retrieve the full network of linear OSM features inside the canton
#    Not working with the `all_private` network because not available in all OSM extracts
#    
# -------------------------------------------------
print("Fetching OSM network … this can take a minute or two")
G_all = ox.graph_from_polygon(polygon, network_type="all", simplify=True)

edges_all = ox.graph_to_gdfs(G_all, nodes=False, edges=True, fill_edge_geometry=True)
# Work in metres ➜ project
edges_all = edges_all.to_crs("EPSG:2056")

# -------------------------------------------------
# 3. Tag pedestrian‑exclusive segments
# -------------------------------------------------
print("Identifying pedestrian-exclusive segments")
ped_highways   = {"footway", "pedestrian", "path", "steps"}

# Note: "cycleway" is not included here as it is not exclusively pedestrian
ped_edges   = edges_all[edges_all["highway"].isin(ped_highways)]



### Compute lenght/width

In [ ]:
# -------------------------------------------------
# 4. Compute linear metrics (metres)
# -------------------------------------------------
print("Computing lengths …")
length_total = edges_all["length"].sum()
length_ped   = ped_edges["length"].sum()


In [ ]:
# Convert width to float if present
def extract_width(row):
    try:
        return float(str(row.get("width")).split(";")[0])
    except (TypeError, ValueError):
        return None

edges_all["width_m"] = edges_all.apply(extract_width, axis=1)

missing_width_count = edges_all["width_m"].isna().sum()
print(f"Number of edges without usable width: {missing_width_count}")

In [ ]:
# -------------------------------------------------
# 5. Very rough surface estimates using typical widths when explicit width=* is missing
#    (Feel free to update these heuristics with local standards!)
# -------------------------------------------------
def estimate_width(row):
    w = row.get("width")
    # Handle list/array widths
    if isinstance(w, (list, tuple)):
        # Try to use the first valid numeric value
        for val in w:
            try:
                return float(str(val).split(";")[0])
            except (ValueError, TypeError):
                continue
    elif pd.notnull(w):
        try:
            return float(str(w).split(";")[0])  # handle "3;3"
        except ValueError:
            pass

    hw = row["highway"]
    if isinstance(hw, list):
        if any(h in ped_highways for h in hw):
            return 2.0
    elif hw in ped_highways:
        return 2.0

    return 3.5  # default generic width

In [ ]:
print("Estimating widths and surfaces …")
edges_all["width_est"]  = edges_all.apply(estimate_width, axis=1)
ped_edges["width_est"]   = ped_edges.apply(estimate_width, axis=1)

surface_total = (edges_all["length"] * edges_all["width_est"]).sum()
surface_ped   = (ped_edges["length"] * ped_edges["width_est"]).sum()

# -------------------------------------------------
# 6. Summaries
# -------------------------------------------------
length_share_ped   = length_ped   / length_total * 100

surface_share_ped   = surface_ped   / surface_total * 100

print("\n=======================  RESULTS  =======================")
print(f"Total network length       : {length_total:,.0f} m")
print(f" – Pedestrian‑dedicated len.: {length_ped:,.0f} m  ({length_share_ped:.2f} %)")
print(f"Total network surface       : {surface_total/1e6:,.2f} ha")
print(f" – Pedestrian‑dedicated surf: {surface_ped/1e6:,.2f} ha ({surface_share_ped:.2f} %)")
print("========================================================\n")


In [ ]:
ped_edges['width_est'].value_counts()

### 1.1 One line/ sidewalk

In [ ]:
ped_highways   = {"footway", "pedestrian", "path", "steps", "cycleway"}

# Add "cycleway" even if not included here as it is not exclusively pedestrian
ped_edges   = edges_all[edges_all["highway"].isin(ped_highways)]

In [ ]:
#check if the CRS is set correctly
print(boundary.crs)
print(edges_all.crs)
print(ped_edges.crs)

In [ ]:
#correct the CRS if not epsg2056
if boundary.crs.to_epsg() != 2056:
    print("Reprojecting boundary to EPSG:2056")
    boundary = boundary.to_crs("EPSG:2056")


In [ ]:
#check the geometry validity
print("Checking geometry validity:")
print(len(boundary))
print(len(edges_all))
print(len(ped_edges))

print(boundary.is_valid.all())
print(edges_all.is_valid.all())
print(ped_edges.is_valid.all())

In [ ]:
# Reproject to EPSG:4326 for web mapping
edges_all_latlon = edges_all.to_crs(epsg=4326)
ped_edges_latlon = ped_edges.to_crs(epsg=4326)

# Create base map centered on Geneva
print('Creating map')
center = ped_edges_latlon.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")

'''# Add all edges in black
folium.GeoJson(
    edges_all_latlon,
    name="All edges",
    style_function=lambda x: {
        "color": "black",
        "weight": 0.5,
        "opacity": 0.8
    }
).add_to(m)'''

# Add pedestrian edges in blue
print('Mapping pedestrian edges')
folium.GeoJson(
    ped_edges_latlon,
    name="Pedestrian paths",
    style_function=lambda x: {
        "color": "#3489db",
        "weight": 2,
        "opacity": 1
    }
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Display
#m 


### 1.2 One surface/ sidewalk

In [ ]:
ped_surfaces = ped_edges.copy()
ped_surfaces["geometry"] = ped_surfaces.buffer(ped_surfaces["width_est"], cap_style=2)


In [ ]:
# Reproject to EPSG:4326 for folium
ped_surfaces_wsg = ped_surfaces.to_crs(epsg=4326)

# Create the folium map
center = ped_surfaces_wsg.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")

# Add surface polygons
folium.GeoJson(
    ped_surfaces_wsg,
    name="Sidewalk surfaces",
    style_function=lambda x: {
        "fillColor": "#9ac4ed",
        "color": "#3489db",
        "weight": 0.5,
        "fillOpacity": 0.6
    }
).add_to(m)

folium.LayerControl().add_to(m)
#m

## 2. Roads' centerline 

In [ ]:
# Fetch road centerlines for Geneva city
area_name = "Geneva, Switzerland"

print(f"Downloading road network for: {area_name}")

ox.settings.timeout = 180
ox.settings.retry_count = 3
ox.settings.overpass_settings = '[out:json][timeout:180]'

custom_filter = (
    '["highway"]["highway"!~"proposed|construction|abandoned|platform|raceway"]'
)

G = ox.graph_from_place(area_name, custom_filter=custom_filter, simplify=True)

# Convert to edges GeoDataFrame
edges = ox.graph_to_gdfs(G, nodes=False, edges=True, fill_edge_geometry=True)
edges = edges.to_crs(epsg=2056)

In [ ]:
print("Edges GeoDataFrame created with the following columns:")
print(edges.columns)
print(edges.crs)

In [ ]:
tags = {"highway": "footway", "footway": "sidewalk"}
sidewalk_ways = ox.features_from_place(area_name, tags=tags)
sidewalk_ways = sidewalk_ways.to_crs(epsg=2056)

In [ ]:
# Buffer roads by 6 meters and spatially join with footway=sidewalk
edges_buffered = edges.copy()
edges_buffered["geometry"] = edges_buffered.buffer(6)
joined = gpd.sjoin(edges_buffered, sidewalk_ways[["geometry"]], how="left", predicate="intersects")

# Assign sidewalk presence
edges["sidewalk"] = "no"
edges.loc[joined.index, "sidewalk"] = "mapped_nearby"
edges["has_sidewalk"] = edges["sidewalk"] == "mapped_nearby"


In [ ]:
# Define walkable road types in CH
walkable_highways = {
    "primary", "secondary", "tertiary",
    "unclassified", "residential", "living_street",
    "pedestrian", "path", "track", "road", "cycleway"
}

# Filter for legal + physically accessible roads
pedestrian_network = edges[
    (edges["highway"].isin(walkable_highways)) &
    (edges["has_sidewalk"])
].copy()


In [ ]:
# Optional: keep only relevant columns
#pedestrian_network = pedestrian_network[["geometry", "length", "highway", "name", "sidewalk", "has_sidewalk"]]
print(f"Final pedestrian road segments: {len(pedestrian_network)}")


In [ ]:
# Filter for footway
footway = edges[edges['highway'] == 'footway'].copy()

#filter cycleway: piste cyclable séparée
cycleway = edges[edges['highway'] =='cycleway'].copy()

# living streets
living_street = edges[edges['highway'] == 'living_street'].copy()


In [ ]:
edges_wgs = edges.to_crs(epsg=4326)
pedestrian_network_wgs = pedestrian_network.to_crs(epsg=4326)
footway_wgs = footway.to_crs(epsg=4326)
cycleway_wgs = cycleway.to_crs(epsg=4326)
living_street_wgs = living_street.to_crs(epsg=4326)

# Center the map on Geneva
center = pedestrian_network_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")

print("All roads")
folium.GeoJson(
    edges_wgs,
    name="All Roads",
    style_function=lambda feature: {
        "color": "black",
        "weight": 0.5,
        "opacity": 0.8
    }
).add_to(m)

'''print("cycleways")
folium.GeoJson(
    cycleway_wgs,
    name="Cycleways",
    style_function=lambda feature: {
        "color": "#28B4A8",
        "weight": 2,
        "opacity": 1
    }
).add_to(m)'''

print("Pedestrian network with sidewalks")
folium.GeoJson(
    pedestrian_network_wgs,
    name="Pedestrian Network (with sidewalk)",
    style_function=lambda feature: {
        "color": "#3489db",
        "weight": 2,
        "opacity": 1
    },
    tooltip=folium.GeoJsonTooltip(fields=["highway", "name"], aliases=["Type", "Street"])
).add_to(m)

'''print("Footways")
folium.GeoJson(
    footway_wgs,
    name="Footways",
    style_function=lambda feature: {
        "color": "#96C8A6",
        "weight": 2,
        "opacity": 1
    },
    tooltip=folium.GeoJsonTooltip(fields=["highway", "name"], aliases=["Type", "Street"])
).add_to(m)'''

'''print("living streets")
folium.GeoJson(
    living_street_wgs,
    name="Living Streets",
    style_function=lambda feature: {
        "color": "#F39C12",
        "weight": 2,
        "opacity": 1
    }
).add_to(m)'''


folium.LayerControl().add_to(m)
#m 

Add footway from parks into pedestrian network

In [ ]:
# 1. Buffer the pedestrian road segments by 6 meters
print("Buffering pedestrian road segments by 6 meters")
centerline_buffers = pedestrian_network.copy()
centerline_buffers["geometry"] = centerline_buffers.buffer(6)

centerline_buffers = centerline_buffers.reset_index(drop=True)
centerline_buffers["road_id"] = centerline_buffers.index

# spatial join : 'road_id_right'
footway = edges[edges['highway'] == 'footway'].copy()
footway = footway.reset_index()


print("Spatial join footways with road buffers")
footways_with_road_proximity = gpd.sjoin(
    footway,
    centerline_buffers[["geometry", "road_id"]],
    how="left",
    predicate="intersects"
)
print(footways_with_road_proximity.head())
# Now use 'road_id' to filter those NOT matched
park_paths = footways_with_road_proximity[footways_with_road_proximity["road_id"].isna()]
park_paths = park_paths.drop(columns=["road_id"])

print(park_paths.head())


In [ ]:
# Reset index to bring u, v, key as columns
pedestrian_network = pedestrian_network.reset_index()

In [ ]:
for col in pedestrian_network.columns:
    if col not in park_paths.columns:
        park_paths[col] = np.nan
park_paths = park_paths[pedestrian_network.columns]

# Now concatenate
pedestrian_network_withparks = pd.concat([pedestrian_network, park_paths], ignore_index=True)

## 3. Area for pedestrians

In [ ]:
area_name = "Geneva, Switzerland"

# exclude highway:pedestrian + area: yes because double info
tags_surface = {
    "leisure": ["park"],
    "landuse": ["recreation_ground"]
    #"highway": "pedestrian",
    #"area": "yes"
}

# Download polygon features with those tags
print(f"Downloading surface features for: {area_name}")
area_features = ox.features_from_place(area_name, tags=tags_surface)

# Filter to polygons only (not points/lines)
print("Filtering to polygon features")
surface_areas = area_features[area_features.geometry.type.isin(["Polygon", "MultiPolygon"])]
surface_areas = surface_areas.to_crs(epsg=2056)

# Keep only rows that are either publicly accessible or have no access restriction
print("Filtering for publicly accessible areas")
if "access" in surface_areas.columns:
    surface_areas = surface_areas[~surface_areas["access"].isin(["private", "no"])]

In [ ]:
#column names of the surface_areas GeoDataFrame
relevant_columns = ["geometry", "leisure", "landuse", "name", "access"]
surface_areas = surface_areas[relevant_columns]


In [ ]:
surface_areas['leisure'].value_counts()

In [ ]:
surface_areas['landuse'].value_counts()

In [ ]:
# Already have pedestrian_network (as GeoDataFrame of LineStrings)
# Combine both types in a single GeoDataFrame if needed
pedestrian_lines = pedestrian_network[["geometry"]].copy()
pedestrian_lines["type"] = "line"

surface_areas = surface_areas[["geometry"]].copy()
surface_areas["type"] = "area"

print("Combining pedestrian lines and surface areas")
# Ensure both have the same CRS
pedestrian_lines = pedestrian_lines.to_crs(epsg=2056)
surface_areas = surface_areas.to_crs(epsg=2056)
pedestrian_full = pd.concat([pedestrian_lines, surface_areas], ignore_index=True)


In [ ]:
# Reproject to latlon
print("Reprojecting to WGS84 for web mapping")
pedestrian_full_wsg = pedestrian_full.to_crs(epsg=4326)
center = pedestrian_full_wsg.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")

# Add line layer
print("Adding pedestrian paths layer")
folium.GeoJson(
    pedestrian_full_wsg[pedestrian_full_wsg["type"] == "line"],
    name="Pedestrian paths",
    style_function=lambda x: {"color": "#3489db", "weight": 2}
).add_to(m)

# Add area layer
print("Adding walkable areas layer")
folium.GeoJson(
    pedestrian_full_wsg[pedestrian_full_wsg["type"] == "area"],
    name="Walkable areas",
    style_function=lambda x: {"color": "#F8D27D", "fillOpacity": 0.4}
).add_to(m)

folium.LayerControl().add_to(m)
#m

## 4. Final map (network options)

In [ ]:
edges_wgs = edges.to_crs(epsg=4326)
pedestrian_network_withparks_wgs = pedestrian_network_withparks.to_crs(epsg=4326)
pedestrian_network_wgs = pedestrian_network.to_crs(epsg=4326)
pedestrian_full_wsg = pedestrian_full.to_crs(epsg=4326)
#pedestrian_network_ME_wsg=pedestrian_network_ME.to_crs(epsg=4326)

# Center the map on Geneva
center = pedestrian_network_withparks_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")
print("All roads")
folium.GeoJson(
    edges_wgs,
    name="Réseau routier",
    style_function=lambda feature: {
        "color": "black",
        "weight": 0.5,
        "opacity": 0.8
    }
).add_to(m)

print("Pedestrian network with parks")
folium.GeoJson(
    pedestrian_network_withparks_wgs,
    name="Réseau piéton: ligne centrée avec parcs",
    style_function=lambda feature: {
        "color": "#3489db",
        "weight": 2,
        "opacity": 1
    },
    tooltip=folium.GeoJsonTooltip(fields=["highway", "name"], aliases=["Type", "Street"])
).add_to(m)

print("Réseau piéton: ligne centrée")
folium.GeoJson(
    pedestrian_network_wgs,
    name="Réseau piéton: ligne centrée",
    style_function=lambda feature: {
        "color": "#3489db",
        "weight": 2,
        "opacity": 1
    },
    tooltip=folium.GeoJsonTooltip(fields=["highway", "name"], aliases=["Type", "Street"])
).add_to(m)

# Add pedestrian edges in blue
print('Pedestrian network: both sides')
folium.GeoJson(
    ped_edges_latlon,
    name="Réseau piéton: 1 ligne par trottoir",
    style_function=lambda x: {
        "color": "#3489db",
        "weight": 2,
        "opacity": 1
    }
).add_to(m)


# Add surface polygons
'''print("Adding sidewalk surfaces")
folium.GeoJson(
    ped_surfaces_wsg,
    name="Réseau piéton: surface de trottoir",
    style_function=lambda x: {
        "fillColor": "#9ac4ed",
        "color": "#3489db",
        "weight": 0.5,
        "fillOpacity": 0.6
    }
).add_to(m)'''

# Add area layer
print("Adding walkable areas layer")
folium.GeoJson(
    pedestrian_full_wsg[pedestrian_full_wsg["type"] == "area"],
    name="Surface piétonne",
    style_function=lambda x: {"fillColor": "#fce8be", "color": "#F8D27D", "weight": 0.5, "fillOpacity": 0.6}
).add_to(m)

print("Adding pedestrian network ME")
folium.GeoJson(
    pedestrian_network_ME_wsg,
    name="Réseau piéton ME",
    style_function=lambda x: {
        "color": "#E76F51",
        "weight": 2,
        "opacity": 1
    }
).add_to(m)


folium.LayerControl().add_to(m)
#m

# Save the map to an HTML file
path = "/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-1"
os.chdir(path)
output_file = "geneva_pedestrian_networks_test.html"
m.save(output_file)

## 5. Street segmentation + save file

Geometry = line

In [ ]:
#import network
print("Loading pedestrian segments, replace file path -->")
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/shp_geneva_pedestrian_edges_all'
pedestrian_network_ME = gpd.read_file(f"{file_path}/geneva_pedestrian_edges_all.shp")
pedestrian_network_ME = pedestrian_network_ME.to_crs(epsg=2056)

In [ ]:

def split_linestring(geom, segment_length):
    """Split a LineString into segments of a given length."""
    if geom.length <= segment_length:
        return [geom]
    
    segments = []
    start = 0.0
    while start < geom.length:
        end = min(start + segment_length, geom.length)
        seg = substring(geom, start, end)
        segments.append(seg)
        start = end
    return segments

# Desired segment length in meters
segment_length = 50

# Store the results
split_rows = []

print("Splitting LineStrings into fixed-length segments...")
# replace name of the GeoDataFrame with the network you want to split
gdf_to_split = pedestrian_network_ME

for idx, row in gdf_to_split.iterrows():
    geom = row.geometry
    if geom.is_empty or not isinstance(geom, LineString):
        continue
    
    segments = split_linestring(geom, segment_length)
    for seg in segments:
        new_row = row.copy()
        new_row.geometry = seg
        new_row["length"] = seg.length
        split_rows.append(new_row)

# New GeoDataFrame
pedestrian_segments = gpd.GeoDataFrame(split_rows, crs=pedestrian_network.crs)

# Add a unique segment ID
print("Assigning unique segment IDs...")
pedestrian_segments.reset_index(drop=True, inplace=True)
pedestrian_segments["segment_id"] = pedestrian_segments.index.astype(str).str.zfill(6)  # e.g., '000001'

In [ ]:
#save to gpkg file
output_file = "step1_pedestrian_segments.gpkg"
pedestrian_segments.to_file(output_file, driver="GPKG")